# 🎭 ChatGLM3-6B 角色扮演聊天机器人

**注意**：6B 模型较大，请按顺序逐个 cell 运行。

如遇内存不足，建议改用 `06.聊天机器人.ipynb`（Qwen3-0.6B，仅需 1.2GB）。

In [1]:
# ===== Cell 1: 加载模型（只运行一次，4-bit 量化版）=====
import os
import subprocess
import torch
import gc
import gradio as gr
from functools import partial
from threading import Thread
from transformers import AutoTokenizer, AutoModel, TextIteratorStreamer, BitsAndBytesConfig

# 自动创建 X: 盘，映射到项目根目录（解决 SentencePiece 不支持中文路径的问题）
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists('X:\\'):
    subprocess.run(['subst', 'X:', project_root], check=False)
    print(f'已创建 X: 盘 -> {project_root}')

model_path = 'X:\\6.transformers\\models\\chatgml3-6B'

# 4-bit 量化配置：6B 模型只需 ~3.8GB 显存，全部放入 GPU，无需 CPU offload
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算时用 bf16，提升速度
    bnb_4bit_use_double_quant=True,         # 二级量化，更省显存
    bnb_4bit_quant_type='nf4',               # NF4 量化，质量更好
)

# 释放旧模型
if 'model' in dir():
    del model
    gc.collect()
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    quantization_config=quantization_config,
).eval()
print(f'模型已加载，设备：{next(model.parameters()).device}')
print(f'显存已用：{torch.cuda.memory_allocated()/1024**2:.1f} MB')



S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 200/200 [00:15<00:00, 13.22it/s]


模型已加载，设备：cuda:0
显存已用：3833.0 MB


In [5]:
# ===== Cell 2: 定义角色和聊天函数 =====
ROLE_PRESETS = {
    '🤖 默认助手': '你是一个有帮助的AI助手，请用中文回答用户的问题。',
    '🌐 翻译官': '你是一个专业的中英翻译官。用户说中文你就翻译成英文，用户说英文你就翻译成中文。只输出翻译结果。',
    '💻 程序员': '你是一个经验丰富的程序员。请用代码和简洁的解释回答编程问题。',
    '🩺 医生': '你是一个有同理心的医生。请给出专业的健康建议，最后提醒用户及时就医。',
    '🎭 诗人': '你是一个诗人。请用优美的诗句回答用户的问题。',
    '服务员': '''🍽️ 饭店服务员(带完整菜单): 
你是本店专业、热情、口语简短的饭店服务员，全程自然口语对话，不啰嗦、不用书面语、不用markdown、不刷屏、每次回复简短一句或两句。

【本店完整菜单与价格】
一、招牌特色菜
1.招牌香辣烤鱼 58元
2.秘制红烧肉 42元
3.椒盐排骨 45元
4.辣子鸡 38元
5.蒜蓉小龙虾 68元

二、家常小炒
1.番茄炒蛋 18元
2.青椒土豆丝 16元
3.鱼香肉丝 26元
4.麻婆豆腐 18元
5.手撕包菜 15元
6.小炒黄牛肉 36元

三、汤类
1.紫菜蛋花汤 12元
2.西红柿鸡蛋汤 12元
3.冬瓜排骨汤 28元

四、主食
1.米饭 2元/碗
2.蛋炒饭 15元
3.炒面 14元

五、饮品
1.雪碧、可乐 5元
2.酸梅汤 6元
3.柠檬水 4元

【服务员工作规则】
1.顾客进门主动问好、询问几位用餐、引导入座。
2.顾客点餐时，可以主动推荐招牌菜、热门菜，询问顾客吃不吃辣、有没有忌口、小孩老人。
3.顾客点完菜要简单复述订单，确认无误。
4.顾客可以随时问价格、菜品辣度、食材、出餐速度、能不能打包、有没有包间、能不能开发票。
5.顾客催菜礼貌安抚，并告知马上帮催后厨。
6.顾客觉得菜品问题，态度好、耐心解答、积极处理。
7.结账时报总价，询问扫码还是现金，询问是否打包剩余饭菜。
8.严格按照本店菜单回答，没有的菜坚决不说、不编造。
9.全程口语化、简短自然，像真实线下服务员，不要机器腔、不要长段落、不要列表。

【对话风格】
自然、温柔、接地气、简短、不啰嗦、一问一答。
''',
}
current_system = ROLE_PRESETS['🤖 默认助手']
# 自己维护对话历史（避免 Gradio 6.x 把 content 转成 list 格式导致报错）
chat_history = []


def set_role(role_name):
    """切换角色，清空对话历史"""
    global current_system, chat_history
    current_system = ROLE_PRESETS[role_name]
    chat_history = []
    return [], f'当前角色：{role_name}'


def clear_chat():
    """清空对话历史"""
    global chat_history
    chat_history = []
    return []


def chat_stream(message, history):
    """流式聊天：用全局 chat_history 维护历史，避免 Gradio 格式转换导致报错"""
    if not message.strip():
        yield history
        return

    # 用全局 chat_history 构建 ChatGLM 输入（content 都是 str，不会报错）
    glm_history = [{'role': 'system', 'content': current_system}] + chat_history
    inputs = tokenizer.build_chat_input(message, history=glm_history, role='user')
    inputs = inputs.to(model.device)

    # 用 TextIteratorStreamer 实现流式输出
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        inputs,
        max_new_tokens=256,       # 减小生成长度，加快速度
        do_sample=True,
        top_p=0.8,
        temperature=0.8,
        streamer=streamer,
        use_cache=False,
    )

    # 子线程运行 generate
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 流式返回：history 只用于 Gradio 显示，模型输入用全局 chat_history
    response = ''
    for new_text in streamer:
        response += new_text
        yield history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': response}]

    # 生成完成后，更新全局历史（content 始终是 str）
    chat_history.append({'role': 'user', 'content': message})
    chat_history.append({'role': 'assistant', 'content': response})

    thread.join()


print('角色和聊天函数已定义')



角色和聊天函数已定义


In [6]:
# ===== Cell 3: 启动 Gradio 界面 =====
if 'demo' in dir():
    demo.close()

with gr.Blocks(title="ChatGLM3-6B 角色扮演") as demo:
    gr.Markdown("# 🎭 ChatGLM3-6B 角色扮演聊天机器人")

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(height=500, placeholder="输入消息开始对话...")
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="输入你的问题，按回车发送...",
                    scale=8,
                    show_label=False,
                )
                submit_btn = gr.Button("发送", scale=1, variant="primary")
            clear_btn = gr.Button("🗑️ 清空对话")

        with gr.Column(scale=1, min_width=200):
            gr.Markdown("### 🎭 角色选择")
            role_btns = [gr.Button(name, size="sm") for name in ROLE_PRESETS]
            role_status = gr.Textbox(
                label="当前角色",
                value="🤖 默认助手",
                interactive=False,
            )
            gr.Markdown(
                "💡 **使用说明**\n"
                "1. 点击右侧按钮切换角色\n"
                "2. 输入问题开始对话\n"
                "3. 历史记录会被保留"
            )

    msg_input.submit(chat_stream, [msg_input, chatbot], [chatbot]).then(
        lambda: "", outputs=[msg_input]
    )
    submit_btn.click(chat_stream, [msg_input, chatbot], [chatbot]).then(
        lambda: "", outputs=[msg_input]
    )
    clear_btn.click(clear_chat, outputs=[chatbot])
    for btn, role_name in zip(role_btns, ROLE_PRESETS):
        btn.click(fn=partial(set_role, role_name), outputs=[chatbot, role_status])

demo.queue().launch(prevent_thread_lock=True, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
# ===== Cell 4: 停止 Gradio 服务（需要时运行）=====
demo.close()
print("Gradio 服务已关闭")

Closing server running on port: 7860
Gradio 服务已关闭
